# DSPy extract pre-filtered pages

In [1]:
model_config_file = f"../output/models/dspy-extract-filtered-light.json"
output_dir = "../output/dspy-extract"
input_dir = "../input/pkna"
input_filtered = "../output/dspy-extract-identify/*/*.json"

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
import dspy


Character = Literal['uno', 'paperinik', 'other']


class ExtractedLine(BaseModel):
    """A line of dialogue extracted from a comic book page."""
    character: Character = Field(
        description="The character who said the line."
    )
    text: str = Field(
        description="The text of the dialogue line."
    )


class CharacterDescription(BaseModel):
    """Description of a character in the comic."""
    name: Character = Field(
        description="The name of the character."
    )
    description: str = Field(
        description="A description and biography of the character."
    )
    model_config = {
        "frozen": True,
    }


class DialogueExtraction(dspy.Signature):
    """Extract dialogues from a comic book page.

    The dialogues are expected to be in the form of text bubbles, typically found in comic books."""

    characters: list[CharacterDescription] = dspy.InputField(
        desc="List of character descriptions we care about."
    )
    page: dspy.Image = dspy.InputField(desc="Comic book page image")
    dialogue: list[ExtractedLine] = dspy.OutputField(
        desc="List of dialogues extracted from the page, each with a character name and text."
    )


class ExtractorModule(dspy.Module):
    """Extract dialogues from a comic book page."""

    def __init__(self, characters: list[CharacterDescription]):
        self.characters = characters
        self.extractor = dspy.ChainOfThought(DialogueExtraction)
        self.normalize = dspy.Predict(
            dspy.make_signature(
                signature='text -> normalized',
                instructions="Normalize text by using normal caps instead of all caps, remove line-break hyphens, and accented letters instead of apostrophes when appropriate.",
            )
        )

    def forward(self, page: dspy.Image) -> dspy.Prediction:
        extracted = self.extractor(
            characters=self.characters, page=page).dialogue
        normalized = [
            self.normalize(text=line.text).normalized
            for line in extracted
        ]
        return dspy.Prediction(
            dialogue=[
                ExtractedLine(character=line.character, text=normalized_text)
                for line, normalized_text in zip(extracted, normalized)
            ],
        )

In [3]:
import os

def load_character(name: Character) -> CharacterDescription:
    """Load a character description from the environment variable."""
    p = os.path.join('../input/bios', f'{name}.md')
    with open(p, 'r', encoding='utf-8') as f:
        description = f.read()
    return CharacterDescription(name=name, description=description)


def init_model() -> ExtractorModule:
    """Initialize the extractor module with characters."""
    characters = [
        load_character('uno'),
        load_character('paperinik'),
        CharacterDescription(
            name='other',
            description='Any other character in the comic book.'
        ),
    ]
    return ExtractorModule(characters=characters)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-flash',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    max_tokens=65535,
)
dspy.configure(
    lm=lm,
    track_usage=True,
)

extractor = init_model()
extractor.load(model_config_file)

In [5]:
from dataclasses import dataclass

@dataclass
class ExtractedPage:
    input_path: str
    extracted: list[ExtractedLine]
    processing_time: float = 0.0
    model_path: str = model_config_file

    def to_dict(self):
        return {
            'input_path': self.input_path,
            'extracted': {
                'dialogue': [
                    {
                        'character': line.character,
                        'text': line.text
                    } for line in self.extracted
                ]
            },
            'processing_time': self.processing_time,
            'model_path': self.model_path
        }

In [6]:
# Figure out which pages need extraction
from pathlib import Path
from glob import glob
import json


def get_pages_with_uno() -> list[Path]:
    res = []
    for p in glob(input_filtered):
        with open(p, 'r') as f:
            data = json.load(f)
            if data['extracted']['uno_present']:
                res.append(Path(data['input_path']))
    return res

input_paths = sorted(get_pages_with_uno())
len(input_paths), len(glob(input_filtered))

(518, 3316)

In [ ]:
import json
import time

def extract_from_page(input_path: Path, output_path: Path) -> None:
    if output_path.exists():
        return

    img = dspy.Image.from_file(str(input_path))
    start_time = time.time()
    extracted = extractor(img)
    assert isinstance(extracted, dspy.Prediction)
    processing_time = time.time() - start_time
    extracted_page = ExtractedPage(
        input_path=str(input_path),
        extracted=extracted.dialogue,
        processing_time=processing_time,
    )
    with open(output_path, 'w') as f:
        json.dump(extracted_page.to_dict(), f, ensure_ascii=False, indent=2)

In [8]:
import os
from tqdm.notebook import tqdm


for page_path in tqdm(input_paths, desc="Processing pages", unit="page"):
    pkna = page_path.parent.stem
    os.makedirs(os.path.join(output_dir, pkna), exist_ok=True)
    output_file = Path(output_dir) / pkna / f"{page_path.stem}.json"
    extract_from_page(page_path, output_file)

Processing pages:   0%|          | 0/518 [00:00<?, ?page/s]